# YOLO11 Face Fine-Tuning on RunPod

This notebook is a clean RunPod-ready version of the detection fine-tuning flow.

It assumes:
- you will either upload `widerface_yolo.zip` to RunPod storage or download it directly from Google Drive
- the zip extracts to a folder named `widerface_yolo`
- that folder already contains `images/`, `labels/`, and `data.yaml`

This version removes:
- Google Drive mounting
- Kaggle download setup
- raw WIDERFace preprocessing

It only keeps the relevant training flow for a preprocessed YOLO dataset.

## 1. Install dependencies

In [ ]:
!pip install -q ultralytics PyYAML gdown

## 2. Configuration

Update these paths if your RunPod workspace uses different locations.

In [ ]:
from pathlib import Path

DATASET_ZIP = '/workspace/widerface_yolo.zip'
DATASET_ROOT = '/workspace/widerface_yolo'
PROJECT_DIR = '/workspace/yolo11_face_runs'
RUN_NAME = 'yolo11_face_finetune'
BASE_MODEL = 'yolo11n.pt'
IMAGE_SIZE = 640
EPOCHS = 25
BATCH_SIZE = 16
DEVICE = 0

# Set either GOOGLE_DRIVE_URL or GOOGLE_DRIVE_FILE_ID if you want RunPod to download the zip directly.
GOOGLE_DRIVE_URL = ''
GOOGLE_DRIVE_FILE_ID = ''

dataset_zip = Path(DATASET_ZIP)
dataset_root = Path(DATASET_ROOT)
project_dir = Path(PROJECT_DIR)
project_dir.mkdir(parents=True, exist_ok=True)

print('dataset_zip:', dataset_zip)
print('dataset_root:', dataset_root)
print('project_dir:', project_dir)

## 3. Download the dataset zip with `gdown` if needed

Paste your Google Drive sharing link into `GOOGLE_DRIVE_URL` or set `GOOGLE_DRIVE_FILE_ID` above.
If `widerface_yolo.zip` is already present in RunPod storage, this cell will skip the download.

Make sure the Drive link is accessible to anyone with the link.

In [ ]:
import gdown

if dataset_zip.exists():
    print('Zip already exists at:', dataset_zip)
elif GOOGLE_DRIVE_URL:
    gdown.download(url=GOOGLE_DRIVE_URL, output=str(dataset_zip), fuzzy=True)
    print('Downloaded zip to:', dataset_zip)
elif GOOGLE_DRIVE_FILE_ID:
    gdown.download(id=GOOGLE_DRIVE_FILE_ID, output=str(dataset_zip))
    print('Downloaded zip to:', dataset_zip)
else:
    raise ValueError('Provide GOOGLE_DRIVE_URL or GOOGLE_DRIVE_FILE_ID, or place widerface_yolo.zip at DATASET_ZIP first.')

## 4. Unzip the dataset

This cell unzips `widerface_yolo.zip` into the configured RunPod workspace path.

If you already extracted it manually, you can skip this cell.

In [ ]:
import zipfile

if dataset_zip.exists() and not dataset_root.exists():
    with zipfile.ZipFile(dataset_zip, 'r') as zf:
        zf.extractall(dataset_root.parent)
    print('Extracted dataset to:', dataset_root)
else:
    print('Skipping unzip.')
    print('zip exists:', dataset_zip.exists())
    print('dataset root already exists:', dataset_root.exists())

## 5. Fix `data.yaml` for the RunPod path

The uploaded `data.yaml` may still contain an absolute path from another machine.
This cell rewrites the dataset root to the current RunPod location and saves a runtime copy.

In [ ]:
import yaml

source_data_yaml = dataset_root / 'data.yaml'
runtime_data_yaml = dataset_root / 'data.runpod.yaml'

if not source_data_yaml.exists():
    raise FileNotFoundError(f'Missing data.yaml at {source_data_yaml}')

with source_data_yaml.open('r') as f:
    data_config = yaml.safe_load(f)

data_config['path'] = str(dataset_root.resolve())

with runtime_data_yaml.open('w') as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

print('runtime data yaml:', runtime_data_yaml)
print(runtime_data_yaml.read_text())

## 6. Sanity check the dataset

In [ ]:
train_images = dataset_root / 'images' / 'train'
val_images = dataset_root / 'images' / 'val'
train_labels = dataset_root / 'labels' / 'train'
val_labels = dataset_root / 'labels' / 'val'

print('train_images exists:', train_images.exists())
print('val_images exists:', val_images.exists())
print('train_labels exists:', train_labels.exists())
print('val_labels exists:', val_labels.exists())

print('train image count:', len(list(train_images.rglob('*.jpg'))))
print('val image count:', len(list(val_images.rglob('*.jpg'))))
print('train label count:', len(list(train_labels.rglob('*.txt'))))
print('val label count:', len(list(val_labels.rglob('*.txt'))))

## 7. Train YOLO11 on face detection

In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_MODEL)
results = model.train(
    data=str(runtime_data_yaml),
    imgsz=IMAGE_SIZE,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    project=str(project_dir),
    name=RUN_NAME,
    device=DEVICE,
    pretrained=True,
)

results

## 8. Validate the best checkpoint

In [ ]:
best_model_path = project_dir / RUN_NAME / 'weights' / 'best.pt'
print('best_model_path:', best_model_path, best_model_path.exists())

best_model = YOLO(str(best_model_path))
metrics = best_model.val(data=str(runtime_data_yaml), imgsz=IMAGE_SIZE, device=DEVICE)
metrics

## 9. Export a clean final checkpoint

This copies the best model to a predictable filename for the main project.

In [ ]:
import shutil

best_model_path = project_dir / RUN_NAME / 'weights' / 'best.pt'
final_model_path = project_dir / 'yolo_face.pt'

shutil.copy2(best_model_path, final_model_path)
print('final_model_path:', final_model_path)